# Stage-31 pz1 600 deg^2 Backlight Paste Validation

This notebook is a lightweight run sheet for the pz1-only validation.  The expensive work is kept in the CLI driver so each step can be launched independently and rerun without rebuilding earlier products.

In [1]:
from pathlib import Path

repo = Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX')
work = repo / 'notebooks/xDESI/abacus_paste'
driver = work / 'stage31_pz1_backlight_validation.py'
config = work / 'stage31_pz1_cap600.yaml'
selected_config = work / 'stage31_pz1_cap600.selected.yaml'
driver, config, selected_config

(PosixPath('/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py'),
 PosixPath('/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.yaml'),
 PosixPath('/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml'))

## 1. Select the 600 deg^2 cap

This writes a derived YAML with the selected cap center and radius.  The common mask uses DESI pz1, DES shear tomos 1-4, ACT y, ACT T, ACT kappa, and DESI pz1 momentum masks.

In [2]:
cmd = f"python {driver} select-cap --config {config} --output-config {selected_config}"
print(cmd)

python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py select-cap --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.yaml --output-config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml


## 2. Build or dry-run the cap-selected halo catalog

The real run streams Abacus ASDF files from `/mnt/ceph/users/backlight/AbacusBacklight_base_c9999_ph9999/lightcone_halos` and writes one HDF5 catalog under `data/xDESI/processed/abacus_backlight/stage31_pz1_cap600/halos/`.

In [3]:
dry_run = f"python {driver} preprocess --config {selected_config} --dry-run --max-files 1"
real_run = f"python {driver} preprocess --config {selected_config} --overwrite"
print(dry_run)
print(real_run)

python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py preprocess --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --dry-run --max-files 1
python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py preprocess --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --overwrite


## 3. Measure pz1 cap data spectra

Start with `nside=128` smoke tests, then run the production `nside=1024` cap.

In [4]:
smoke_data = f"python {driver} measure-data --config {selected_config} --nside 128 --overwrite"
prod_data = f"python {driver} measure-data --config {selected_config} --nside 1024 --overwrite"
print(smoke_data)
print(prod_data)

python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py measure-data --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 128 --overwrite
python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py measure-data --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 1024 --overwrite


## 4. Build pz1 theory and unresolved correction

This builds pz1-only Stage-31 theory, plus a resolved-only analytic curve with HOD occupation zeroed below `log10(M200c/[Msun/h])=11.0`.

In [5]:
theory_smoke = f"python {driver} theory --config {selected_config} --nside 128"
theory_prod = f"python {driver} theory --config {selected_config} --nside 1024"
print(theory_smoke)
print(theory_prod)

python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py theory --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 128
python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py theory --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 1024


## 5. Pasting hooks

The wrapper exposes split and combine commands.  The cap catalog is reusable, so paste jobs should not reread the full Backlight ASDF tree after preprocessing.

In [6]:
for split in range(4):
    print(f"python {driver} paste-split --config {selected_config} --nside 1024 --split-index {split} --num-splits 4 --overwrite")
print(f"python {driver} combine-maps --config {selected_config} --nside 1024 --num-splits 4 --overwrite")

python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py paste-split --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 1024 --split-index 0 --num-splits 4 --overwrite
python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py paste-split --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 1024 --split-index 1 --num-splits 4 --overwrite
python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py paste-split --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 1024 --split-index 2 --num-splits 4 --overwrite
python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py p

## 6. Plot overlay

Once the theory product path is known, pass it to the plot command.  Add `--sim path/to/sim_measurement.h5` when the pz1 simulation NaMaster product is available.

In [7]:
theory_h5 = '<fill theory h5 path printed by the theory step>'
plot_cmd = f"python {driver} plot --config {selected_config} --nside 1024 --theory {theory_h5}"
print(plot_cmd)

python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_backlight_validation.py plot --config /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/abacus_paste/stage31_pz1_cap600.selected.yaml --nside 1024 --theory <fill theory h5 path printed by the theory step>
